In [1]:
import pandas as pd

# Charge le fichier — adapte le chemin si besoin
df = pd.read_csv("Online Sales Data.csv")

# Vue générale
print("=== DIMENSIONS ===")
print(df.shape)

print("\n=== PREMIÈRES LIGNES ===")
print(df.head())

print("\n=== TYPES DE COLONNES ===")
print(df.dtypes)

print("\n=== VALEURS MANQUANTES ===")
print(df.isnull().sum())

print("\n=== STATISTIQUES DESCRIPTIVES ===")
print(df.describe())

print("\n=== VALEURS UNIQUES PAR COLONNE ===")
for col in df.columns:
    print(f"{col} : {df[col].nunique()} valeurs uniques")

=== DIMENSIONS ===
(240, 9)

=== PREMIÈRES LIGNES ===
   Transaction ID        Date Product Category             Product Name  \
0           10001  2024-01-01      Electronics            iPhone 14 Pro   
1           10002  2024-01-02  Home Appliances         Dyson V11 Vacuum   
2           10003  2024-01-03         Clothing         Levi's 501 Jeans   
3           10004  2024-01-04            Books        The Da Vinci Code   
4           10005  2024-01-05  Beauty Products  Neutrogena Skincare Set   

   Units Sold  Unit Price  Total Revenue         Region Payment Method  
0           2      999.99        1999.98  North America    Credit Card  
1           1      499.99         499.99         Europe         PayPal  
2           3       69.99         209.97           Asia     Debit Card  
3           4       15.99          63.96  North America    Credit Card  
4           1       89.99          89.99         Europe         PayPal  

=== TYPES DE COLONNES ===
Transaction ID        int64
Da

In [2]:

# ÉTAPE 2 — Nettoyage & préparation
# 1. Convertir la colonne Date en format datetime
df['Date'] = pd.to_datetime(df['Date'])

# 2. Extraire des colonnes utiles pour l'analyse
df['Mois'] = df['Date'].dt.month
df['Mois_nom'] = df['Date'].dt.strftime('%B')
df['Trimestre'] = df['Date'].dt.quarter
df['Année'] = df['Date'].dt.year

# 3. Vérifier que tout est bon
print("=== TYPES APRÈS NETTOYAGE ===")
print(df.dtypes)

print("\n=== APERÇU DES NOUVELLES COLONNES ===")
print(df[['Date', 'Mois', 'Mois_nom', 'Trimestre', 'Année']].head(10))

print("\n=== CATÉGORIES DE PRODUITS ===")
print(df['Product Category'].value_counts())

print("\n=== RÉGIONS ===")
print(df['Region'].value_counts())

print("\n=== CHIFFRE D'AFFAIRES TOTAL ===")
print(f"CA total : {df['Total Revenue'].sum():,.2f} $")
print(f"Panier moyen : {df['Total Revenue'].mean():,.2f} $")
print(f"Vente max : {df['Total Revenue'].max():,.2f} $")

=== TYPES APRÈS NETTOYAGE ===
Transaction ID               int64
Date                datetime64[ns]
Product Category            object
Product Name                object
Units Sold                   int64
Unit Price                 float64
Total Revenue              float64
Region                      object
Payment Method              object
Mois                         int32
Mois_nom                    object
Trimestre                    int32
Année                        int32
dtype: object

=== APERÇU DES NOUVELLES COLONNES ===
        Date  Mois Mois_nom  Trimestre  Année
0 2024-01-01     1  January          1   2024
1 2024-01-02     1  January          1   2024
2 2024-01-03     1  January          1   2024
3 2024-01-04     1  January          1   2024
4 2024-01-05     1  January          1   2024
5 2024-01-06     1  January          1   2024
6 2024-01-07     1  January          1   2024
7 2024-01-08     1  January          1   2024
8 2024-01-09     1  January          1   2024
9 

In [3]:
# ============================================
# ÉTAPE 3 — Analyse SQL
# ============================================

import sqlite3

# Créer une base de données en mémoire et y charger le dataframe
conn = sqlite3.connect(':memory:')
df.to_sql('ventes', conn, index=False, if_exists='replace')

print("✅ Base de données créée avec succès !")
print(f"Table 'ventes' chargée avec {len(df)} lignes.\n")

# --- REQUÊTE 1 : CA par catégorie de produit ---
print("=== CA PAR CATÉGORIE ===")
q1 = pd.read_sql_query("""
    SELECT 
        [Product Category] AS Catégorie,
        COUNT(*) AS Nb_ventes,
        ROUND(SUM([Total Revenue]), 2) AS CA_total,
        ROUND(AVG([Total Revenue]), 2) AS Panier_moyen
    FROM ventes
    GROUP BY [Product Category]
    ORDER BY CA_total DESC
""", conn)
print(q1.to_string(index=False))

# --- REQUÊTE 2 : CA mensuel ---
print("\n=== CA MENSUEL ===")
q2 = pd.read_sql_query("""
    SELECT 
        Mois,
        Mois_nom,
        COUNT(*) AS Nb_ventes,
        ROUND(SUM([Total Revenue]), 2) AS CA_mensuel
    FROM ventes
    GROUP BY Mois, Mois_nom
    ORDER BY Mois
""", conn)
print(q2.to_string(index=False))

# --- REQUÊTE 3 : Top 5 produits ---
print("\n=== TOP 5 PRODUITS ===")
q3 = pd.read_sql_query("""
    SELECT 
        [Product Name] AS Produit,
        [Product Category] AS Catégorie,
        COUNT(*) AS Nb_ventes,
        ROUND(SUM([Total Revenue]), 2) AS CA_total
    FROM ventes
    GROUP BY [Product Name], [Product Category]
    ORDER BY CA_total DESC
    LIMIT 5
""", conn)
print(q3.to_string(index=False))

# --- REQUÊTE 4 : CA par région ---
print("\n=== CA PAR RÉGION ===")
q4 = pd.read_sql_query("""
    SELECT 
        Region,
        COUNT(*) AS Nb_ventes,
        ROUND(SUM([Total Revenue]), 2) AS CA_total,
        ROUND(AVG([Total Revenue]), 2) AS Panier_moyen
    FROM ventes
    GROUP BY Region
    ORDER BY CA_total DESC
""", conn)
print(q4.to_string(index=False))

# --- REQUÊTE 5 : Moyen de paiement préféré ---
print("\n=== MOYENS DE PAIEMENT ===")
q5 = pd.read_sql_query("""
    SELECT 
        [Payment Method] AS Paiement,
        COUNT(*) AS Nb_transactions,
        ROUND(SUM([Total Revenue]), 2) AS CA_total
    FROM ventes
    GROUP BY [Payment Method]
    ORDER BY Nb_transactions DESC
""", conn)
print(q5.to_string(index=False))

✅ Base de données créée avec succès !
Table 'ventes' chargée avec 240 lignes.

=== CA PAR CATÉGORIE ===
      Catégorie  Nb_ventes  CA_total  Panier_moyen
    Electronics         40  34982.41        874.56
Home Appliances         40  18646.16        466.15
         Sports         40  14326.52        358.16
       Clothing         40   8128.93        203.22
Beauty Products         40   2621.90         65.55
          Books         40   1861.93         46.55

=== CA MENSUEL ===
 Mois Mois_nom  Nb_ventes  CA_mensuel
    1  January         31    14548.32
    2 February         29    10803.37
    3    March         31    12849.24
    4    April         30    12451.69
    5      May         31     8455.49
    6     June         30     7384.55
    7     July         31     6797.08
    8   August         27     7278.11

=== TOP 5 PRODUITS ===
                  Produit       Catégorie  Nb_ventes  CA_total
      Canon EOS R5 Camera     Electronics          1   3899.99
               LG OLED TV H

In [4]:
# ============================================
# EXPORT POUR POWER BI
# ============================================

# Exporter le dataframe nettoyé avec les nouvelles colonnes
df.to_csv('ventes_nettoyees.csv', index=False, encoding='utf-8-sig')

# Exporter les tables d'agrégation SQL
q1.to_csv('ca_par_categorie.csv', index=False, encoding='utf-8-sig')
q2.to_csv('ca_mensuel.csv', index=False, encoding='utf-8-sig')
q3.to_csv('top5_produits.csv', index=False, encoding='utf-8-sig')
q4.to_csv('ca_par_region.csv', index=False, encoding='utf-8-sig')
q5.to_csv('moyens_paiement.csv', index=False, encoding='utf-8-sig')

print("✅ Fichiers exportés avec succès !")
print("📁 Fichiers créés :")
print("   - ventes_nettoyees.csv (fichier principal)")
print("   - ca_par_categorie.csv")
print("   - ca_mensuel.csv")
print("   - top5_produits.csv")
print("   - ca_par_region.csv")
print("   - moyens_paiement.csv")

✅ Fichiers exportés avec succès !
📁 Fichiers créés :
   - ventes_nettoyees.csv (fichier principal)
   - ca_par_categorie.csv
   - ca_mensuel.csv
   - top5_produits.csv
   - ca_par_region.csv
   - moyens_paiement.csv
